# 04 · Backbone mô hình

Mô hình gồm hai tầng:

1. **VideoMAE-Base** (Kinetics-400), đóng băng: 16 frame → 1 vector 768 chiều (xem notebook 02).
2. **FeatureTransformer**: nhận 18 vector 768 chiều liên tiếp → logits 10 lớp. Đây là classifier được huấn luyện, và cũng là phần được fine-tune ở notebook 05.

Kiến trúc FeatureTransformer: `Linear(768→768)` + `LayerNorm` → positional encoding sin/cos → 2 lớp Transformer encoder (8 head, Pre-LayerNorm) → trung bình theo thời gian → `Linear(768→10)`.

> Lưu ý: `FeatureTransformer` lấy từ repo chính thức OmniFall (`omnifall-experiments/models/feature_transformer.py`, không nằm trong repo này — clone riêng theo README). Notebook chép code vào đây để giải thích từng phần; hàm `from_pretrained` và lớp wrapper cho HuggingFace Trainer được bỏ qua vì pipeline này không dùng.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

## Import

Các thư viện cần cho các cell bên dưới.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer

## Positional encoding

Transformer không biết thứ tự phần tử. Positional encoding cộng vào mỗi vị trí một vector sin/cos có tần số khác nhau, để mô hình phân biệt feature thứ 1 với feature thứ 18.

**`PositionalEncoding`** (chép từ `feature_transformer.py`): tính sẵn bảng `pe` kích thước `(1, max_len, d_model)` và đăng ký làm *buffer* (lưu cùng model nhưng không phải tham số học). `forward` cộng `pe` vào đầu vào theo độ dài chuỗi.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()

        # Compute the positional encodings in log space
        # Using 100 as default max_len to support longer sequences if needed
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        # Register buffer (not a parameter, but part of the module)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, d_model]
        """
        return x + self.pe[:, : x.size(1), :]

## FeatureTransformer

Class được tách làm 3 cell: khai báo các lớp con, khởi tạo trọng số, và forward.

**Khai báo các lớp con (`__init__`):**
- `feature_projection` + `feature_norm`: chiếu feature VideoMAE sang không gian ẩn và chuẩn hoá.
- `pos_encoder`: positional encoding với `max_len = num_features = 18`.
- `transformer_encoder`: 2 lớp encoder, 8 head, FFN rộng `4 × 768`, `norm_first=True` (Pre-LayerNorm, ổn định hơn với mô hình nhỏ).
- `classifier`: lớp tuyến tính ra 10 lớp.

In [ ]:
class FeatureTransformer(nn.Module):
    def __init__(self, feature_dim=2048, hidden_dim=768, num_classes=10, num_layers=2,
                 num_heads=8, dropout=0.1, mlp_ratio=4.0, num_features=3):
        super(FeatureTransformer, self).__init__()
        self.feature_dim = feature_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_features = num_features
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout
        self.mlp_ratio = mlp_ratio

        # Feature projection
        self.feature_projection = nn.Linear(feature_dim, hidden_dim)
        self.feature_norm = nn.LayerNorm(hidden_dim)

        # Add positional encoding
        self.pos_encoder = PositionalEncoding(hidden_dim, max_len=num_features)

        # Create transformer encoder with Pre-LayerNorm for better stability
        encoder_layers = TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=int(hidden_dim * mlp_ratio),
            dropout=dropout,
            batch_first=True,
            norm_first=True,  # Pre-LayerNorm improves stability on small stacks
        )
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers)

        # Classification head
        self.classifier = nn.Linear(hidden_dim, num_classes)

        # Initialize weights
        self.apply(self._init_weights)

**Khởi tạo trọng số:** mọi lớp `Linear` dùng Xavier normal, bias = 0. Cell này định nghĩa hàm rồi gắn vào class.

In [ ]:
def _init_weights(self, module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

FeatureTransformer._init_weights = _init_weights

**Forward:** `(B, 18, 768)` → chiếu + chuẩn hoá → cộng positional encoding → 2 lớp Transformer (mỗi feature "nhìn" 17 feature còn lại) → **trung bình theo thời gian** thành 1 vector → classifier → logits `(B, 10)`.
Đầu ra là dict `{"logits": ...}` cho tương thích với HuggingFace Trainer.

In [ ]:
def forward(self, features):
    # Project features and apply layer normalization
    x = self.feature_projection(features)
    x = self.feature_norm(x)

    # Add positional encoding
    x = self.pos_encoder(x)

    # Apply transformer encoder
    x = self.transformer_encoder(x)

    # Global average pooling over sequence dimension
    x = torch.mean(x, dim=1)

    # Classification
    logits = self.classifier(x)

    return {"logits": logits}

FeatureTransformer.forward = forward

**Ví dụ nhỏ:** tạo model đúng cấu hình của dự án, đếm tham số và chạy thử với dữ liệu ngẫu nhiên.

In [ ]:
model = FeatureTransformer(feature_dim=768, hidden_dim=768, num_classes=10,
                           num_layers=2, num_heads=8, num_features=18).eval()
print("Số tham số:", sum(p.numel() for p in model.parameters()) / 1e6, "triệu")
x = torch.randn(4, 18, 768)          # 4 segment, mỗi segment 18 feature VideoMAE
with torch.no_grad():
    print(model(x)["logits"].shape)  # (4, 10)

## Nạp checkpoint baseline (`tcfall/baseline.py`)

Import cần cho hai hàm bên dưới. `FeatureTransformer` dùng bản đã định nghĩa ở trên, nên không cần clone `omnifall-experiments` để đọc notebook này (nhưng cần file checkpoint để thực sự nạp trọng số).

In [ ]:
import torch
from safetensors.torch import load_file
from tcfall.config import BASELINE_CKPT, FEATURE_DIM, NUM_CLASSES, NUM_FEATURES

**`tcfall/baseline.py` · load_baseline**

Dựng `FeatureTransformer` với cấu hình của dự án và nạp trọng số:
- mặc định: checkpoint-7000 gốc (huấn luyện trên Staged + Syn), file `.safetensors`; khoá có tiền tố `model.` (từ wrapper) được bỏ đi;
- nếu truyền `ckpt` hoặc đặt biến môi trường `TCFALL_BASELINE_CKPT`: nạp bản đã fine-tune trên ItW-train (notebook 05).

`strict=True` để chắc chắn mọi khoá khớp. Hàm import `FeatureTransformer` từ repo `omnifall-experiments`, nên cần clone repo đó trước khi gọi.

In [ ]:
def load_baseline(device: str = "cuda", ckpt: str | None = None) -> FeatureTransformer:
    """Checkpoint gốc (St+Sy). `ckpt` (hoặc biến môi trường TCFALL_BASELINE_CKPT) = file .pt
    state_dict của bản đã fine-tune trên ItW-train (pipeline/s05b_finetune_itw.py)."""
    import os
    model = FeatureTransformer(feature_dim=FEATURE_DIM, hidden_dim=768, num_classes=NUM_CLASSES,
                               num_layers=2, num_heads=8, num_features=NUM_FEATURES)
    ckpt = ckpt or os.environ.get("TCFALL_BASELINE_CKPT")
    if ckpt:
        sd = torch.load(ckpt, map_location="cpu", weights_only=True)
    else:
        sd = load_file(str(BASELINE_CKPT))
        sd = {k[6:] if k.startswith("model.") else k: v for k, v in sd.items()}
    model.load_state_dict(sd, strict=True)
    return model.to(device).eval()

**`tcfall/baseline.py` · predict_logits**

Chạy classifier theo lô 256 chuỗi, trả logits trên CPU. `@torch.no_grad()` tắt tính gradient khi suy luận.

In [ ]:
@torch.no_grad()
def predict_logits(model, seqs: torch.Tensor, batch: int = 256) -> torch.Tensor:
    """seqs (N, 18, 768) -> logits (N, 10)."""
    dev = next(model.parameters()).device
    out = [model(seqs[i:i + batch].to(dev).float())["logits"].float().cpu() for i in range(0, len(seqs), batch)]
    return torch.cat(out, 0)